# Inspect one SWE-bench Verified task

This notebook downloads the benchmark metadata and prints one task's issue and evaluation-test information. It does not run a coding agent or modify a repository, so it is suitable for a Google Colab CPU runtime.

Set `TASK_ID` below when you know the exact task. Otherwise, leave it as `None` and inspect the task at `TASK_INDEX`. The reference solution fields are not printed by default.

In [ ]:
%pip -q install -U datasets

In [ ]:
from datasets import load_dataset

DATASET_NAME = "princeton-nlp/SWE-bench_Verified"
DATASET_SPLIT = "test"

# Use an exact instance_id when you have one; otherwise the first task is selected.
TASK_ID = None
TASK_INDEX = 0

dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT)
print(f"Loaded {len(dataset):,} tasks from {DATASET_NAME!r} ({DATASET_SPLIT!r} split).")


def select_task(rows, task_id=None, task_index=0):
    """Select one task by stable instance_id or by zero-based index."""
    if task_id is not None:
        for row in rows:
            if row["instance_id"] == task_id:
                return row
        raise ValueError(f"No task found with instance_id={task_id!r}")

    if not 0 <= task_index < len(rows):
        raise IndexError(f"TASK_INDEX must be between 0 and {len(rows) - 1}")
    return rows[task_index]


task = select_task(dataset, task_id=TASK_ID, task_index=TASK_INDEX)
print(f"Selected task: {task['instance_id']}")

In [ ]:
required_fields = ("instance_id", "repo", "base_commit", "problem_statement")
missing_fields = [field for field in required_fields if not task.get(field)]
assert not missing_fields, f"The selected row is missing required fields: {missing_fields}"


def print_field(title, value):
    print(f"\n{title}")
    print("-" * len(title))
    if value is None or value == "":
        print("(not provided)")
    elif isinstance(value, (list, tuple)):
        print("\n".join(f"- {item}" for item in value))
    else:
        print(value)

print_field("Task metadata", {
    field: task.get(field)
    for field in (
        "instance_id",
        "repo",
        "version",
        "base_commit",
        "environment_setup_commit",
        "created_at",
    )
})
print_field("Problem statement", task["problem_statement"].strip())
print_field("FAIL_TO_PASS tests", task.get("FAIL_TO_PASS"))
print_field("PASS_TO_PASS tests", task.get("PASS_TO_PASS"))

if task.get("hints_text"):
    print_field("Hints", task["hints_text"].strip())

print("\nAvailable fields:")
print(", ".join(sorted(task.keys())))
print("\nReference fields intentionally not printed: patch, test_patch")

## Clone the complete repository for this task

The repository is cloned into the current runtime, then checked out at the task's base_commit. The output below shows where the full repository is located. In Colab, the default path is usually under /content and is temporary unless you copy it to Google Drive.

In [ ]:
from pathlib import Path
import subprocess


def run_git(arguments, cwd=None, capture_output=True):
    return subprocess.run(
        ["git", *arguments],
        cwd=cwd,
        check=True,
        text=True,
        capture_output=capture_output,
    )


repo_url = f"https://github.com/{task['repo']}.git"
base_commit = task["base_commit"]
safe_instance_id = "".join(
    character if character.isalnum() or character in "._-" else "_"
    for character in task["instance_id"]
)
repo_parent = Path.cwd() / "swebench_repos"
repo_dir = repo_parent / safe_instance_id
repo_parent.mkdir(parents=True, exist_ok=True)

if repo_dir.exists():
    if not (repo_dir / ".git").exists():
        raise FileExistsError(f"Target exists but is not a Git repository: {repo_dir}")
    origin_url = run_git(["config", "--get", "remote.origin.url"], cwd=repo_dir).stdout.strip()
    if origin_url not in {repo_url, repo_url.removesuffix(".git")}:
        raise ValueError(f"Existing clone has unexpected origin: {origin_url}")
    print(f"Using existing clone: {repo_dir}")
else:
    print(f"Cloning {repo_url} ...")
    run_git(["clone", repo_url, str(repo_dir)], capture_output=False)

commit_exists = subprocess.run(
    ["git", "cat-file", "-e", f"{base_commit}^{{commit}}"],
    cwd=repo_dir,
).returncode == 0
if not commit_exists:
    run_git(["fetch", "--quiet", "origin", base_commit], cwd=repo_dir)

run_git(["checkout", "--quiet", "--detach", base_commit], cwd=repo_dir)
actual_commit = run_git(["rev-parse", "HEAD"], cwd=repo_dir).stdout.strip()
tracked_files = run_git(["ls-files"], cwd=repo_dir).stdout.splitlines()

print(f"\\nTask: {task['instance_id']}")
print(f"Repository URL: {repo_url}")
print(f"Repository path on this runtime: {repo_dir.resolve()}")
print(f"Checked-out commit: {actual_commit}")
print(f"Matches task base_commit: {actual_commit == base_commit}")
print(f"Tracked files: {len(tracked_files):,}")

max_files_to_print = 100
print(f"\\nFirst {min(max_files_to_print, len(tracked_files))} tracked files:")
print("\\n".join(f"- {file_name}" for file_name in tracked_files[:max_files_to_print]))
if len(tracked_files) > max_files_to_print:
    print(f"... {len(tracked_files) - max_files_to_print:,} more files are present in the repository.")

## Try another task

1. Copy the printed `instance_id` into `TASK_ID`.
2. Set `TASK_INDEX = 0` (the index is ignored when `TASK_ID` is set).
3. Re-run the selection and inspection cells.

This is an inspection step only. Executing the actual SWE-bench repair task requires an isolated repository environment and the official evaluation harness; it is not performed here.